# Project Title: Real-Time Data Extraction and Machine Learning for Optimized Uber Ride Booking using MLflow

### Objective:
- To develop a system that extracts real-time data from Uber, analyzes it using machine learning algorithms, and provides users with optimized Uber ride booking options. The application will allow users to input their pickup location, destination, and pickup time through a Streamlit interface, and it will suggest the best time to book an Uber ride based on the predicted fare.

### Project Scope and Applications:
- End-to-End Development: The project involves real-time data extraction, machine learning for fare prediction, and delivering results via a Streamlit application.
- Real-World Relevance: The system will optimize Uber ride booking by suggesting the cheapest fare times, potentially saving users money and time.


#### This step comes after the creation and running of webscrapping in main.py (Data collection started from 2024-09-10 and is currently running still for 1 hour intervals between 7 am and 11 pm for 30 different routes

# 1.0 Setting up MySql Connection

In [36]:
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy import text

# # Database connection parameters
# username = 'root'
# password = 'expert789'
# host = 'localhost'
# port = '3306'
# database = 'uber'

# # Create a database connection
# engine = create_engine(f'mysql+mysqlconnector://{username}:{password}@{host}:{port}/{database}')

# # Create a connection
# with engine.connect() as connection:
#     # Execute the query
#     result = connection.execute(text("SELECT * FROM uber_details"))
#     df = pd.DataFrame(result.fetchall(), columns=result.keys())


In [37]:

# # Create a connection
# with engine.connect() as connection:
#     # Execute the query
#     result = connection.execute(text("SELECT * FROM uber_details"))
#     df = pd.DataFrame(result.fetchall(), columns=result.keys())


In [38]:
path = 'D:\\GUVI\\Assesments\\Uber demand price Prediction\\data\\justadf.csv'
df = pd.read_csv(path)


In [39]:
df

,route_from,route_to,ride_type,ride_max_persons,hour,day_of_week,ride_waiting_time,ride_time_minutes,ride_price
0,Chennai Citi Centre,Chennai Lighthouse,Go Sedan,4.0,17,1,6.000000,7.983333,147.060000
1,Chennai Citi Centre,Chennai Lighthouse,Moto,1.0,17,1,5.333333,7.983333,35.416667
2,Chennai Citi Centre,Chennai Lighthouse,Premier,4.0,17,1,6.000000,8.316667,199.183333
3,Chennai Citi Centre,Chennai Lighthouse,Uber Auto,3.0,17,1,1.333333,7.650000,70.000000
4,Chennai Citi Centre,Chennai Lighthouse,Uber Go,4.0,17,1,6.000000,8.650000,143.010000
...,...,...,...,...,...,...,...,...,...
17377,Semmozhi Poonga,Sai Baba Temple Mylapore,Moto,1.0,20,0,3.000000,16.133333,45.650000
17378,Semmozhi Poonga,Sai Baba Temple Mylapore,Premier,4.0,20,0,3.000000,16.133333,226.700000
17379,Semmozhi Poonga,Sai Baba Temple Mylapore,Uber Auto,3.0,20,0,2.000000,16.133333,156.730000
17380,Semmozhi Poonga,Sai Baba Temple Mylapore,Uber Go,4.0,20,0,6.000000,18.133333,143.850000


# 2.0 Exploratory Data Analysis

In [41]:
df

,route_from,route_to,ride_type,ride_max_persons,hour,day_of_week,ride_waiting_time,ride_time_minutes,ride_price
0,Chennai Citi Centre,Chennai Lighthouse,Go Sedan,4.0,17,1,6.000000,7.983333,147.060000
1,Chennai Citi Centre,Chennai Lighthouse,Moto,1.0,17,1,5.333333,7.983333,35.416667
2,Chennai Citi Centre,Chennai Lighthouse,Premier,4.0,17,1,6.000000,8.316667,199.183333
3,Chennai Citi Centre,Chennai Lighthouse,Uber Auto,3.0,17,1,1.333333,7.650000,70.000000
4,Chennai Citi Centre,Chennai Lighthouse,Uber Go,4.0,17,1,6.000000,8.650000,143.010000
...,...,...,...,...,...,...,...,...,...
17377,Semmozhi Poonga,Sai Baba Temple Mylapore,Moto,1.0,20,0,3.000000,16.133333,45.650000
17378,Semmozhi Poonga,Sai Baba Temple Mylapore,Premier,4.0,20,0,3.000000,16.133333,226.700000
17379,Semmozhi Poonga,Sai Baba Temple Mylapore,Uber Auto,3.0,20,0,2.000000,16.133333,156.730000
17380,Semmozhi Poonga,Sai Baba Temple Mylapore,Uber Go,4.0,20,0,6.000000,18.133333,143.850000


In [64]:
# print(df['ride_request_date'].min())
# print(df['ride_request_date'].max())


In [107]:
# from summarytools import dfSummary

In [ ]:
# dfSummary(df)

In [87]:
df.head(1)

,route_from,route_to,ride_type,ride_max_persons,hour,day_of_week,ride_waiting_time,ride_time_minutes,ride_price
0,Chennai Citi Centre,Chennai Lighthouse,Go Sedan,4.0,17,1,6.0,7.983333,147.06


In [88]:
df.dtypes

route_from            object
route_to              object
ride_type             object
ride_max_persons     float64
hour                   int64
day_of_week            int64
ride_waiting_time    float64
ride_time_minutes    float64
ride_price           float64
dtype: object

# 3. Data Preprocessing

In [90]:
# Convert ride_request_time to proper datetime
df['ride_request_time'] = pd.to_datetime(df['ride_request_date'] + ' ' + df['ride_request_time'])

# Drop unnecessary columns
df = df.drop(columns=['id', 'ride_reaching_time'])

# Convert 'ride_time' to total seconds and then to minutes
df['ride_time_seconds'] = pd.to_timedelta(df['ride_time']).dt.total_seconds()
df['ride_time_minutes'] = df['ride_time_seconds'] / 60

# Create 'ride_time_request_clean' by rounding the ride_request_time to the nearest hour
df['ride_time_request_clean'] = df['ride_request_time'].dt.round('H').dt.time

# Group by 'route_from', 'route_to', 'ride_request_date', 'ride_time_request_clean', and 'ride_type'
grouped = df.groupby(['route_from', 'route_to', 'ride_request_date', 'ride_time_request_clean', 'ride_type'], as_index=False).agg({
    'ride_price': 'mean',
    'ride_max_persons': 'mean',
    'ride_waiting_time': 'mean',
    'ride_time_minutes': 'mean'  # Use ride_time_minutes for time-related calculations
})



KeyError: 'ride_request_date'

In [ ]:
grouped

In [ ]:
grouped.dtypes

In [ ]:
# Convert 'ride_request_date' to datetime (date only)
grouped['ride_request_date'] = pd.to_datetime(grouped['ride_request_date'], format='%Y-%m-%d')

# Convert 'ride_time_request_clean' to a string (if it's not already)
grouped['ride_time_request_clean'] = grouped['ride_time_request_clean'].astype(str)

# Combine 'ride_request_date' and 'ride_time_request_clean'
grouped['ride_datetime'] = pd.to_datetime(grouped['ride_request_date'].astype(str) + ' ' + grouped['ride_time_request_clean'])

# Extract the hour from the combined datetime
grouped['hour'] = grouped['ride_datetime'].dt.hour
grouped['day_of_week'] = grouped['ride_datetime'].dt.dayofweek

In [ ]:
grouped

In [ ]:
# Load your cleaned dataset
df = pd.DataFrame(grouped)

In [ ]:
df= df[['route_from','route_to','ride_type','ride_max_persons','hour','day_of_week','ride_waiting_time','ride_time_minutes','ride_price']]
df

# 4.0 Model Selection and Training

In [109]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Assuming your DataFrame is already loaded into `df`

# Step 1: Split the data into features (X) and targets (y)
X = df.drop(columns=['ride_price', 'ride_waiting_time', 'ride_time_minutes'])
y_price = df['ride_price']
y_waiting_time = df['ride_waiting_time']
y_ride_time = df['ride_time_minutes']

# Step 2: Train-Test Split
X_train, X_test, y_train_price, y_test_price = train_test_split(X, y_price, test_size=0.2, random_state=42)
_, _, y_train_waiting, y_test_waiting = train_test_split(X, y_waiting_time, test_size=0.2, random_state=42)
_, _, y_train_time, y_test_time = train_test_split(X, y_ride_time, test_size=0.2, random_state=42)

# Define numerical and categorical columns
numerical_features = ['ride_max_persons', 'hour', 'day_of_week']
categorical_features = ['route_from', 'route_to', 'ride_type']

# Create preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# Step 3: Create pipelines for each target
price_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42))
])

waiting_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42))
])

time_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42))
])

# Step 4: Fit the models
price_pipeline.fit(X_train, y_train_price)
waiting_pipeline.fit(X_train, y_train_waiting)
time_pipeline.fit(X_train, y_train_time)

# Step 5: Predict on the test data
price_predictions = price_pipeline.predict(X_test)
waiting_predictions = waiting_pipeline.predict(X_test)
time_predictions = time_pipeline.predict(X_test)

# Step 6: Evaluate the models
def evaluate_model(true, predicted, metric_name="Model"):
    print(f"--- {metric_name} ---")
    print("Mean Absolute Error (MAE):", mean_absolute_error(true, predicted))
    print("Mean Squared Error (MSE):", mean_squared_error(true, predicted))
    print("R² Score:", r2_score(true, predicted))
    print()

# Evaluate price prediction model
evaluate_model(y_test_price, price_predictions, "Random Forest Price Prediction Model")

# Evaluate waiting time prediction model
evaluate_model(y_test_waiting, waiting_predictions, "Random Forest Waiting Time Prediction Model")

# Evaluate ride time prediction model
evaluate_model(y_test_time, time_predictions, "Random Forest Ride Time Prediction Model")


--- Random Forest Price Prediction Model ---
Mean Absolute Error (MAE): 9.988839353849103
Mean Squared Error (MSE): 371.49167398346134
R² Score: 0.9689393955955695

--- Random Forest Waiting Time Prediction Model ---
Mean Absolute Error (MAE): 0.3824662064998562
Mean Squared Error (MSE): 0.39420552439842776
R² Score: 0.9295044444594749

--- Random Forest Ride Time Prediction Model ---
Mean Absolute Error (MAE): 0.6320974578979327
Mean Squared Error (MSE): 0.6685754614426112
R² Score: 0.9908743896218605



In [110]:
import xgboost as xgb

# Create pipelines for each target
xg_price_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', xgb.XGBRegressor(objective='reg:squarederror', random_state=42))
])

xg_waiting_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', xgb.XGBRegressor(objective='reg:squarederror', random_state=42))
])

xg_time_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', xgb.XGBRegressor(objective='reg:squarederror', random_state=42))
])

# Step 4: Fit the models
xg_price_pipeline.fit(X_train, y_train_price)
xg_waiting_pipeline.fit(X_train, y_train_waiting)
xg_time_pipeline.fit(X_train, y_train_time)

# Step 5: Predict on the test data
price_predictions = xg_price_pipeline.predict(X_test)
waiting_predictions = xg_waiting_pipeline.predict(X_test)
time_predictions = xg_time_pipeline.predict(X_test)

# Step 6: Evaluate the models
def evaluate_model(true, predicted, metric_name="Model"):
    print(f"--- {metric_name} ---")
    print("Mean Absolute Error (MAE):", mean_absolute_error(true, predicted))
    print("Mean Squared Error (MSE):", mean_squared_error(true, predicted))
    print("R² Score:", r2_score(true, predicted))
    print()

# Evaluate price prediction model
evaluate_model(y_test_price, price_predictions, "XGBoost Price Prediction Model")

# Evaluate waiting time prediction model
evaluate_model(y_test_waiting, waiting_predictions, "XGBoost Waiting Time Prediction Model")

# Evaluate ride time prediction model
evaluate_model(y_test_time, time_predictions, "XGBoost Ride Time Prediction Model")


--- XGBoost Price Prediction Model ---
Mean Absolute Error (MAE): 9.986590159927001
Mean Squared Error (MSE): 273.16059368533735
R² Score: 0.9771609063310599

--- XGBoost Waiting Time Prediction Model ---
Mean Absolute Error (MAE): 0.6093194111717386
Mean Squared Error (MSE): 0.6864234757526041
R² Score: 0.8772472701059107

--- XGBoost Ride Time Prediction Model ---
Mean Absolute Error (MAE): 0.5913222087712239
Mean Squared Error (MSE): 0.6152340680856367
R² Score: 0.9916024641637416



In [111]:
from sklearn.linear_model import LinearRegression

# Create pipelines for each target
lr_price_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression()
)
])

lr_waiting_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression()
)
])

lr_time_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression()
)
])

# Fit Linear Regression models
lr_price_pipeline.fit(X_train, y_train_price)
lr_waiting_pipeline.fit(X_train, y_train_waiting)
lr_time_pipeline.fit(X_train, y_train_time)

# Predict and evaluate Linear Regression models
price_lr_predictions = lr_price_pipeline.predict(X_test)
waiting_lr_predictions = lr_waiting_pipeline.predict(X_test)
time_lr_predictions = lr_time_pipeline.predict(X_test)

# Evaluate Linear Regression models
evaluate_model(y_test_price, price_lr_predictions, "Linear Regression Price Prediction Model")
evaluate_model(y_test_waiting, waiting_lr_predictions, "Linear Regression Waiting Time Prediction Model")
evaluate_model(y_test_time, time_lr_predictions, "Linear Regression Ride Time Prediction Model")


--- Linear Regression Price Prediction Model ---
Mean Absolute Error (MAE): 33.72205073304059
Mean Squared Error (MSE): 2209.297850534934
R² Score: 0.8152795032760836

--- Linear Regression Waiting Time Prediction Model ---
Mean Absolute Error (MAE): 1.309132574475667
Mean Squared Error (MSE): 2.8820733532640146
R² Score: 0.4846004188882411

--- Linear Regression Ride Time Prediction Model ---
Mean Absolute Error (MAE): 3.010745847558439
Mean Squared Error (MSE): 16.662556296219385
R² Score: 0.7725671888480382



# 5.0 Hyperparameter tuning

In [113]:
## Random Forest Performs the best, performing hyper tuning to see best one

from sklearn.model_selection import RandomizedSearchCV

# Define parameter distributions
param_distributions = {
    'model__n_estimators': [50, 100, 200, 300],
    'model__max_depth': [None, 10, 20, 30, 40],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4]
}
# Create RandomizedSearchCV objects with verbose parameter
price_random_search = RandomizedSearchCV(price_pipeline, param_distributions, n_iter=1, cv=2, n_jobs=-1, scoring='neg_mean_squared_error', random_state=42, verbose=3)
waiting_random_search = RandomizedSearchCV(waiting_pipeline, param_distributions, n_iter=1, cv=2, n_jobs=-1, scoring='neg_mean_squared_error', random_state=42, verbose=3)
time_random_search = RandomizedSearchCV(time_pipeline, param_distributions, n_iter=1, cv=2, n_jobs=-1, scoring='neg_mean_squared_error', random_state=42, verbose=3)

# Fit RandomizedSearchCV objects
price_random_search.fit(X_train, y_train_price)
waiting_random_search.fit(X_train, y_train_waiting)
time_random_search.fit(X_train, y_train_time)

# Print best parameters and scores
print("Best parameters for price model:", price_random_search.best_params_)
print("Best score for price model:", -price_random_search.best_score_)
print("Best parameters for waiting time model:", waiting_random_search.best_params_)
print("Best score for waiting time model:", -waiting_random_search.best_score_)
print("Best parameters for ride time model:", time_random_search.best_params_)
print("Best score for ride time model:", -time_random_search.best_score_)

Fitting 2 folds for each of 1 candidates, totalling 2 fits
Fitting 2 folds for each of 1 candidates, totalling 2 fits
Fitting 2 folds for each of 1 candidates, totalling 2 fits
Best parameters for price model: {'model__n_estimators': 200, 'model__min_samples_split': 5, 'model__min_samples_leaf': 4, 'model__max_depth': 20}
Best score for price model: 751.1702408376416
Best parameters for waiting time model: {'model__n_estimators': 200, 'model__min_samples_split': 5, 'model__min_samples_leaf': 4, 'model__max_depth': 20}
Best score for waiting time model: 0.9101100865118965
Best parameters for ride time model: {'model__n_estimators': 200, 'model__min_samples_split': 5, 'model__min_samples_leaf': 4, 'model__max_depth': 20}
Best score for ride time model: 1.283745342036314


In [114]:
# Pipeline with best fit parameters
# Create pipelines for each targets
price_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=200, 
                                    min_samples_split =5,
                                    min_samples_leaf=4, 
                                    max_depth=20, 
                                    random_state=42))
])

waiting_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=200, 
                                    min_samples_split =5,
                                    min_samples_leaf=4, 
                                    max_depth=20, 
                                    random_state=42))
])

time_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=200, 
                                    min_samples_split =5,
                                    min_samples_leaf=4, 
                                    max_depth=20, 
                                    random_state=42))
])

# Fit the models
price_pipeline.fit(X_train, y_train_price)
waiting_pipeline.fit(X_train, y_train_waiting)
time_pipeline.fit(X_train, y_train_time)

# Predict on the test data
price_predictions = price_pipeline.predict(X_test)
waiting_predictions = waiting_pipeline.predict(X_test)
time_predictions = time_pipeline.predict(X_test)

# Evaluate the models
def evaluate_model(true, predicted, metric_name="Model"):
    print(f"--- {metric_name} ---")
    print("Mean Absolute Error (MAE):", mean_absolute_error(true, predicted))
    print("Mean Squared Error (MSE):", mean_squared_error(true, predicted))
    print("R² Score:", r2_score(true, predicted))
    print()

# Evaluate price prediction model
evaluate_model(y_test_price, price_predictions, "Random Forest Price Prediction Model")

# Evaluate waiting time prediction model
evaluate_model(y_test_waiting, waiting_predictions, "Random Forest Waiting Time Prediction Model")

# Evaluate ride time prediction model
evaluate_model(y_test_time, time_predictions, "Random Forest Ride Time Prediction Model")


--- Random Forest Price Prediction Model ---
Mean Absolute Error (MAE): 12.768593201476193
Mean Squared Error (MSE): 498.5060326648156
R² Score: 0.9583196616285046

--- Random Forest Waiting Time Prediction Model ---
Mean Absolute Error (MAE): 0.5345000260213328
Mean Squared Error (MSE): 0.5987619941830166
R² Score: 0.8929237242910318

--- Random Forest Ride Time Prediction Model ---
Mean Absolute Error (MAE): 0.6995823396459554
Mean Squared Error (MSE): 0.8441495341746398
R² Score: 0.98847792030365



In [115]:
# Seems like the default parameters perform well on the test data, 
# so we will choose that 

In [116]:
# Let's save our model

In [117]:
import joblib

# Save the pipelines
joblib.dump(price_pipeline, 'price_pipeline.pkl')
joblib.dump(waiting_pipeline, 'waiting_pipeline.pkl')
joblib.dump(time_pipeline, 'time_pipeline.pkl')


['time_pipeline.pkl']

In [120]:
# df.to_csv('data/justadf.csv',index=False) #To get unique values for streamlit

# Now that model is selected and saved, we can go ahead and start creating our Streamlit Application